# CMP461 Pattern Recognition - Midterm Project (+15 Bonus Points)
## Phase 4: Capsule Network (CapsNet) - ULTIMATE REFINEMENT

**Author:** Efe Yaşar  
**Student ID:** 210408030 

--- 

### Optimization: Margin Loss and Enhanced Feature Maps
### Optimizasyon: Margin Loss ve Geliştirilmiş Öznitelik Haritaları
This version implements the specialized **Margin Loss** from the original Hinton paper and utilizes **One-Hot Encoding** for class separation. We also added a **Learning Rate Scheduler** to ensure stable convergence on our 3000-image dataset.

Bu versiyon, orijinal Hinton makalesindeki özel **Margin Loss**'u uygular ve sınıf ayrımı için **One-Hot Encoding** kullanır. Ayrıca 3000 resimlik veri setimizde kararlı bir yakınsama sağlamak için bir **Öğrenme Hızı Çizelgesi** ekledik.

In [1]:
# --- STEP 1: Core Definitions and Custom Margin Loss ---
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, backend as K, callbacks
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from tensorflow.keras.utils import to_categorical

RESULTS_DIR = "results/capsnet"
os.makedirs(RESULTS_DIR, exist_ok=True)

def squash(vectors, axis=-1):
    s_squared_norm = tf.reduce_sum(tf.square(vectors), axis, keepdims=True)
    scale = s_squared_norm / (1 + s_squared_norm) / tf.sqrt(s_squared_norm + K.epsilon())
    return scale * vectors

def margin_loss(y_true, y_pred):
    """
    Specialized Margin Loss for CapsNet. Separates class probabilities by force.
    CapsNet için özel Margin Loss. Sınıf olasılıklarını zorla ayırır.
    """
    L = y_true * tf.square(tf.maximum(0., 0.9 - y_pred)) + \
        0.5 * (1 - y_true) * tf.square(tf.maximum(0., y_pred - 0.1))
    return tf.reduce_mean(tf.reduce_sum(L, axis=-1))

print("✅ Margin Loss engine ready. / Margin Loss motoru hazır.")

/Users/efe/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


✅ Margin Loss engine ready. / Margin Loss motoru hazır.


In [2]:
# --- STEP 2: Advanced Capsule Layer ---
class CapsuleLayer(layers.Layer):
    def __init__(self, num_capsule, dim_capsule, routings=3, **kwargs):
        super(CapsuleLayer, self).__init__(**kwargs)
        self.num_capsule = num_capsule
        self.dim_capsule = dim_capsule
        self.routings = routings

    def build(self, input_shape):
        self.input_num_capsule = input_shape[1]
        self.input_dim_capsule = input_shape[2]
        self.W = self.add_weight(shape=[self.input_num_capsule, self.num_capsule, 
                                        self.dim_capsule, self.input_dim_capsule],
                                 initializer='glorot_uniform', name='W')
        self.built = True

    def call(self, inputs):
        inputs_hat = tf.einsum('bic,ijkc->bijk', inputs, self.W)
        b = tf.zeros(shape=[tf.shape(inputs_hat)[0], self.input_num_capsule, self.num_capsule])
        
        for i in range(self.routings):
            c = tf.nn.softmax(b, axis=-1)
            outputs = squash(tf.einsum('bij,bijk->bjk', c, inputs_hat))
            if i < self.routings - 1:
                b += tf.einsum('bjk,bijk->bij', outputs, inputs_hat)
        return outputs

print("🧠 Neural Routing Logic Optimized.")

🧠 Neural Routing Logic Optimized.


In [3]:
# --- STEP 3: Building Refined CapsNet Architecture ---
def build_refined_capsnet():
    inputs = layers.Input(shape=(128, 128, 3))
    
    # Deeper Feature Extraction / Daha Derin Öznitelik Çıkarımı
    x = layers.Conv2D(256, 9, strides=1, padding='valid', activation='relu')(inputs)
    
    # Primary Capsules: 32 channels of 8D capsules
    x = layers.Conv2D(256, 9, strides=2, padding='valid')(x)
    x = layers.Reshape(target_shape=[-1, 8])(x)
    x = layers.Lambda(squash)(x)
    
    # Digit Capsules: Vector of length 3 for our classes
    digit_caps = CapsuleLayer(num_capsule=3, dim_capsule=16, routings=3)(x)
    
    # Output is magnitude of capsules / Çıkış kapsüllerin büyüklüğüdür
    out = layers.Lambda(lambda x: tf.sqrt(tf.reduce_sum(tf.square(x), axis=-1) + K.epsilon()))(digit_caps)

    model = models.Model(inputs=inputs, outputs=out)
    # Using refined hyperparameters / Geliştirilmiş hiperparametreler
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001), 
                  loss=margin_loss, metrics=['accuracy'])
    return model

caps_model = build_refined_capsnet()
caps_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 120, 120, 256)  │        62,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 56, 56, 256)    │     5,308,672 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 100352, 8)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, 100352, 8)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ capsule_layer (CapsuleLayer)    │ (None, 3, 16)          │    38,535,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda_1 (Lambda)               │ (None, 3)              │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 43,906,304 (167.49 MB)

 Trainable params: 43,906,304 (167.49 MB)

 Non-trainable params: 0 (0.00 B)

In [4]:
# --- STEP 4: Training with One-Hot Formatting ---
print("📊 Preparing One-Hot data... / One-Hot verileri hazırlanıyor...")
data = np.load("preprocessed_data.npz")
X_train, y_train = data["X_train"], to_categorical(data["y_train"], 3)
X_val, y_val = data["X_val"], to_categorical(data["y_val"], 3)
X_test, y_test = data["X_test"], to_categorical(data["y_test"], 3)

# Callbacks for professional training control
lr_reducer = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)

print("🚀 Launching Ultimate CapsNet Marathon... / Nihai CapsNet Maratonu Başlıyor...")
history = caps_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=32,
    callbacks=[lr_reducer],
    verbose=1
)

# Evaluation and Logging
test_loss, test_acc = caps_model.evaluate(X_test, y_test, verbose=0)
print(f"\n🔥 Final Refined CapsNet Accuracy: {test_acc:.4f}")

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1); plt.plot(history.history['accuracy'], label='Train'); plt.plot(history.history['val_accuracy'], label='Val'); plt.legend(); plt.title("Accuracy")
plt.subplot(1, 2, 2); plt.plot(history.history['loss'], label='Train'); plt.plot(history.history['val_loss'], label='Val'); plt.legend(); plt.title("Loss")
plt.savefig(f"{RESULTS_DIR}/ultimate_capsnet_history.png")
plt.show()

y_pred = np.argmax(caps_model.predict(X_test), axis=1)
y_true = np.argmax(y_test, axis=1)
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', xticklabels=["Cats", "Dogs", "Snakes"], yticklabels=["Cats", "Dogs", "Snakes"])
plt.title("Refined CapsNet CM")
plt.savefig(f"{RESULTS_DIR}/ultimate_capsnet_cm.png")
plt.show()

caps_model.save("models/Efe_Yasar_CapsNet_Ultimate.keras")
print("✨ All developmental phases concluded. Preparing for final report!")

📊 Preparing One-Hot data... / One-Hot verileri hazırlanıyor...
🚀 Launching Ultimate CapsNet Marathon... / Nihai CapsNet Maratonu Başlıyor...
Epoch 1/15
 5/66 ━━━━━━━━━━━━━━━━━━━━ 9:16 9s/step - accuracy: 0.2966 - loss: 0.7261

KeyboardInterrupt: 